# 20.21B — Auditoria estrutural, caminhos, schemas e universo controlado de Hamiltonianos

Notebook autônomo para AWS/SageMaker. Localiza automaticamente `20_22A_yahoo_dataset_v2`, confirma caminhos, inventaria arquivos e colunas, audita registry/raw/processed/splits/instances/manifests, cria manifests persistentes, gera IDs estáveis e separa train/calibration/test sem vazamento por `parent_instance_id`.

In [ ]:
%pip install -q pandas numpy pyarrow scipy scikit-learn tqdm

In [ ]:

from pathlib import Path
import os, json, hashlib, itertools, math, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
pd.set_option('display.max_columns',200)
pd.set_option('display.width',220)
pd.set_option('display.max_colwidth',120)
DATASET_BASENAME='20_22A_yahoo_dataset_v2'
VERSION='v2'
STAGE_NAME='20_21B'
OUTPUT_SUBDIR='06_20_21B_hamiltonian_universe'
K_VALUES=[3,4,5]
MAX_COMBINATIONS_PER_K=50000
RANDOM_SEED=20260828
TRAIN_FRAC,CAL_FRAC,TEST_FRAC=0.70,0.15,0.15
assert abs(TRAIN_FRAC+CAL_FRAC+TEST_FRAC-1)<1e-12


## 1. Localização automática do dataset e confirmação dos caminhos

In [ ]:

def candidate_roots():
    cwd=Path.cwd().resolve()
    cands=[cwd,cwd.parent,cwd.parent.parent,Path.home(),Path.home()/"Downloads",Path.home()/"SageMaker",Path('/home/sagemaker-user'),Path('/home/sagemaker-user/SageMaker'),Path('/home/ec2-user/SageMaker'),Path('/root'),Path('/tmp')]
    out=[]; seen=set()
    for p in cands:
        try:p=p.resolve()
        except:pass
        if str(p) not in seen:
            seen.add(str(p)); out.append(p)
    return out

def find_dataset_root(name=DATASET_BASENAME):
    hits=[]
    for base in candidate_roots():
        if not base.exists(): continue
        d=base/name
        if d.is_dir(): hits.append(d.resolve())
        try:
            for child in base.iterdir():
                if child.is_dir() and child.name==name: hits.append(child.resolve())
        except: pass
    for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents]:
        if p.name==name and p.is_dir(): hits.append(p)
    uniq=[]; seen=set()
    for h in hits:
        if str(h) not in seen: seen.add(str(h)); uniq.append(h)
    if not uniq:
        raise FileNotFoundError(f"Não encontrei {name}. CWD={Path.cwd().resolve()}")
    if len(uniq)>1:
        print('Múltiplas cópias encontradas:')
        for i,h in enumerate(uniq): print(i,h)
        print('Usando:',uniq[0])
    return uniq[0]

ROOT=find_dataset_root()
DIRS={
 'root':ROOT,
 'registry':ROOT/'00_registry',
 'raw':ROOT/'01_raw',
 'processed':ROOT/'02_processed',
 'splits':ROOT/'03_splits',
 'instances':ROOT/'04_instances',
 'manifests':ROOT/'05_manifests',
 'stage':ROOT/OUTPUT_SUBDIR,
}
for k,p in DIRS.items():
    if k=='stage': p.mkdir(parents=True,exist_ok=True)
    elif not p.exists(): raise FileNotFoundError(f'Pasta obrigatória ausente: {k} -> {p}')
print('='*100); print('CAMINHOS RESOLVIDOS'); print('='*100)
for k,p in DIRS.items(): print(f'{k:12s} -> {p.resolve()}')


In [ ]:

path_manifest=pd.DataFrame([{'logical_name':k,'absolute_path':str(p.resolve()),'exists':p.exists(),'is_dir':p.is_dir()} for k,p in DIRS.items()])
display(path_manifest)
PATH_MANIFEST_FILE=DIRS['stage']/f'path_manifest__{STAGE_NAME}__{VERSION}.csv'
path_manifest.to_csv(PATH_MANIFEST_FILE,index=False)
print('Salvo:',PATH_MANIFEST_FILE.resolve())


## 2. Inventário completo de arquivos

In [ ]:

rows=[]
for logical_dir,folder in DIRS.items():
    if logical_dir in {'root','stage'}: continue
    for f in sorted(folder.rglob('*')):
        if f.is_file(): rows.append({'logical_dir':logical_dir,'relative_path':str(f.relative_to(ROOT)),'filename':f.name,'suffix':f.suffix.lower(),'size_bytes':f.stat().st_size,'size_mb':f.stat().st_size/(1024**2),'absolute_path':str(f.resolve())})
file_inventory=pd.DataFrame(rows)
display(file_inventory)
display(file_inventory.groupby('logical_dir').agg(files=('filename','size'),size_mb=('size_mb','sum')).reset_index())
INVENTORY_FILE=DIRS['stage']/f'file_inventory__{STAGE_NAME}__{VERSION}.csv'
file_inventory.to_csv(INVENTORY_FILE,index=False)


In [ ]:

EXPECTED_REGISTRY={
 'asset_registry':DIRS['registry']/f'asset_registry__{VERSION}.csv',
 'asset_eligibility':DIRS['registry']/f'asset_eligibility__{VERSION}.csv',
 'return_availability':DIRS['registry']/f'return_availability__{VERSION}.csv',
}
for name,p in EXPECTED_REGISTRY.items(): print(f'{name:22s}: {"OK" if p.exists() else "AUSENTE"} -> {p.resolve()}')
missing=[str(p) for p in EXPECTED_REGISTRY.values() if not p.exists()]
if missing: raise FileNotFoundError('Arquivos obrigatórios ausentes:
'+'
'.join(missing))


## 3. Auditoria detalhada de schemas e colunas

In [ ]:

def read_table(path):
    if path.suffix.lower()=='.csv': return pd.read_csv(path)
    if path.suffix.lower() in {'.parquet','.pq'}: return pd.read_parquet(path)
    raise ValueError(path)

def detect_date_like(df):
    for c in ['Date','date','Datetime','datetime','timestamp','Timestamp','time','Time']:
        if c in df.columns:
            parsed=pd.to_datetime(df[c],errors='coerce')
            if parsed.notna().mean()>0.8: return c,parsed
    try:
        idx=pd.to_datetime(df.index,errors='coerce')
        if pd.Series(idx).notna().mean()>0.8: return '__index__',idx
    except: pass
    return None,None

def inspect_table(path):
    df=read_table(path); ds,dates=detect_date_like(df)
    out={'relative_path':str(path.relative_to(ROOT)),'shape_rows':len(df),'shape_cols':df.shape[1],'columns_json':json.dumps(list(map(str,df.columns)),ensure_ascii=False),'dtypes_json':json.dumps({str(c):str(df[c].dtype) for c in df.columns},ensure_ascii=False),'date_source':ds,'date_min':None,'date_max':None,'total_na':int(df.isna().sum().sum())}
    if dates is not None:
        v=pd.Series(dates).dropna()
        if len(v): out['date_min']=str(v.min()); out['date_max']=str(v.max())
    return out

tabular_paths=[Path(p) for p in file_inventory.loc[file_inventory['suffix'].isin(['.csv','.parquet','.pq']),'absolute_path']]
schema_rows=[]
for p in tqdm(tabular_paths,desc='Auditando schemas'):
    try:schema_rows.append(inspect_table(p))
    except Exception as e:schema_rows.append({'relative_path':str(p.relative_to(ROOT)),'read_error':repr(e)})
schema_manifest=pd.DataFrame(schema_rows)
display(schema_manifest)
SCHEMA_MANIFEST_FILE=DIRS['stage']/f'schema_manifest__{STAGE_NAME}__{VERSION}.csv'
schema_manifest.to_csv(SCHEMA_MANIFEST_FILE,index=False)
print('Salvo:',SCHEMA_MANIFEST_FILE.resolve())


In [ ]:

for name,path in EXPECTED_REGISTRY.items():
    print('
'+'='*100); print(name,'->',path.resolve())
    df=read_table(path); print('shape:',df.shape); print('colunas:')
    for i,c in enumerate(df.columns): print(f'[{i:02d}] {c} :: {df[c].dtype}')
    display(df.head(10))


## 4. Resolução explícita de aliases de colunas

In [ ]:

ALIASES={
 'asset_id':['asset_id','canonical_id','symbol_id','asset_name','name','unified_name','ticker_canonical','canonical_name'],
 'ticker':['ticker','symbol','yf_ticker','yahoo_ticker','Ticker'],
 'eligible':['eligible','is_eligible','train_eligible','coverage_eligible'],
 'coverage':['coverage','train_coverage','coverage_train','coverage_fraction'],
 'market':['market','country','region','exchange_group'],
 'asset_class':['asset_class','class','asset_type','instrument_type'],
}
def resolve_col(df,logical_name,required=False):
    for cand in ALIASES.get(logical_name,[]):
        if cand in df.columns:
            print(f'{logical_name:14s} -> {cand}'); return cand
    if required: raise KeyError(f"Não encontrei coluna para {logical_name}. Colunas={list(df.columns)}")
    print(f'{logical_name:14s} -> NÃO RESOLVIDO'); return None
registry_df=read_table(EXPECTED_REGISTRY['asset_registry'])
eligibility_df=read_table(EXPECTED_REGISTRY['asset_eligibility'])
availability_df=read_table(EXPECTED_REGISTRY['return_availability'])
REG_ASSET_ID=resolve_col(registry_df,'asset_id'); REG_TICKER=resolve_col(registry_df,'ticker'); REG_MARKET=resolve_col(registry_df,'market'); REG_CLASS=resolve_col(registry_df,'asset_class')
ELIG_ASSET_ID=resolve_col(eligibility_df,'asset_id'); ELIG_TICKER=resolve_col(eligibility_df,'ticker'); ELIG_FLAG=resolve_col(eligibility_df,'eligible'); ELIG_COVERAGE=resolve_col(eligibility_df,'coverage')


In [ ]:

def choose_key(df,asset_id_col=None,ticker_col=None,table_name='table'):
    if asset_id_col in df.columns if asset_id_col else False: return asset_id_col
    if ticker_col in df.columns if ticker_col else False: return ticker_col
    c0=df.columns[0]
    if df[c0].nunique(dropna=True)==len(df):
        warnings.warn(f'{table_name}: usando primeira coluna única {c0} como chave.'); return c0
    raise KeyError(f'{table_name}: sem chave única. Colunas={list(df.columns)}')
REG_KEY=choose_key(registry_df,REG_ASSET_ID,REG_TICKER,'asset_registry')
ELIG_KEY=choose_key(eligibility_df,ELIG_ASSET_ID,ELIG_TICKER,'asset_eligibility')
print('REG_KEY=',REG_KEY,' ELIG_KEY=',ELIG_KEY)
if registry_df[REG_KEY].duplicated().any(): raise RuntimeError('Duplicatas no registry')
if eligibility_df[ELIG_KEY].duplicated().any(): raise RuntimeError('Duplicatas em eligibility')


## 5. Auditoria 1:1 registry ↔ raw

In [ ]:

raw_files=sorted(DIRS['raw'].glob('*.parquet'))
print('Ativos no registry:',len(registry_df)); print('Linhas eligibility:',len(eligibility_df)); print('Arquivos raw parquet:',len(raw_files))
raw_names=pd.DataFrame({'raw_file':[f.name for f in raw_files],'raw_stem':[f.stem for f in raw_files],'absolute_path':[str(f.resolve()) for f in raw_files]})
display(raw_names.head(50))
registry_values=set(registry_df[REG_KEY].astype(str))
matches=[]
for value in sorted(registry_values):
    containing=[f.name for f in raw_files if value.lower() in f.name.lower()]
    matches.append({'registry_key':value,'n_raw_name_matches':len(containing),'raw_name_matches':' | '.join(containing)})
registry_raw_name_audit=pd.DataFrame(matches)
display(registry_raw_name_audit)
display(registry_raw_name_audit['n_raw_name_matches'].value_counts().sort_index())
registry_raw_name_audit.to_csv(DIRS['stage']/f'registry_raw_name_audit__{STAGE_NAME}__{VERSION}.csv',index=False)


In [ ]:

raw_audit_rows=[]
for f in tqdm(raw_files,desc='Auditando raw'):
    df=pd.read_parquet(f); ds,dates=detect_date_like(df)
    row={'raw_file':f.name,'absolute_path':str(f.resolve()),'rows':len(df),'cols':df.shape[1],'columns':' | '.join(map(str,df.columns)),'total_na':int(df.isna().sum().sum()),'na_fraction':float(df.isna().mean().mean()) if df.size else np.nan,'date_source':ds,'date_min':None,'date_max':None}
    if dates is not None:
        v=pd.Series(dates).dropna()
        if len(v): row['date_min']=v.min(); row['date_max']=v.max()
    raw_audit_rows.append(row)
raw_audit=pd.DataFrame(raw_audit_rows)
display(raw_audit)
display(raw_audit.groupby(['rows','cols']).size().reset_index(name='n_files'))
RAW_AUDIT_FILE=DIRS['stage']/f'raw_audit__{STAGE_NAME}__{VERSION}.csv'; raw_audit.to_csv(RAW_AUDIT_FILE,index=False)


## 6. Elegibilidade real e tabela canônica

In [ ]:

if ELIG_FLAG:
    flag=eligibility_df[ELIG_FLAG]
    if flag.dtype!=bool: flag=flag.astype(str).str.lower().isin(['true','1','yes','y','sim'])
    eligible_df=eligibility_df.loc[flag].copy(); print(f'Elegíveis: {int(flag.sum())}/{len(flag)}')
else:
    eligible_df=eligibility_df.copy(); print('ATENÇÃO: sem flag explícita; usando todas as linhas.')
if ELIG_COVERAGE: display(eligibility_df[ELIG_COVERAGE].describe())
display(eligible_df.head(50))


In [ ]:

canonical=registry_df.copy().rename(columns={REG_KEY:'asset_key'}); canonical['asset_key']=canonical['asset_key'].astype(str)
elig_tmp=eligibility_df.copy().rename(columns={ELIG_KEY:'asset_key'}); elig_tmp['asset_key']=elig_tmp['asset_key'].astype(str)
elig_cols=[c for c in elig_tmp.columns if c!='asset_key' and c not in canonical.columns]
canonical=canonical.merge(elig_tmp[['asset_key']+elig_cols],on='asset_key',how='left').sort_values('asset_key').reset_index(drop=True)
canonical.insert(0,'asset_index',np.arange(len(canonical),dtype=int))
CANONICAL_FILE=DIRS['stage']/f'canonical_asset_table__{STAGE_NAME}__{VERSION}.csv'; canonical.to_csv(CANONICAL_FILE,index=False)
print('Tabela canônica:',canonical.shape,'->',CANONICAL_FILE.resolve()); display(canonical)


## 7. Universo controlado de Hamiltonianos com IDs estáveis

In [ ]:

def stable_hash(text,n=16): return hashlib.sha256(text.encode()).hexdigest()[:n]
def parent_instance_id(asset_keys,k): return stable_hash(f'dataset={VERSION}|k={k}|assets='+','.join(sorted(map(str,asset_keys))),20)
eligible_keys=sorted(set(eligible_df[ELIG_KEY].astype(str).tolist() if ELIG_KEY in eligible_df.columns else canonical['asset_key'].astype(str).tolist()))
rng=np.random.default_rng(RANDOM_SEED)
def sample_combinations(items,k,max_n):
    total=math.comb(len(items),k)
    if total<=max_n: return list(itertools.combinations(items,k)),total,'full_enumeration'
    seen=set(); combos=[]
    while len(combos)<max_n:
        idx=tuple(sorted(rng.choice(len(items),size=k,replace=False)))
        if idx in seen: continue
        seen.add(idx); combos.append(tuple(items[i] for i in idx))
    combos.sort(); return combos,total,'deterministic_random_sample'
print('Ativos elegíveis:',len(eligible_keys))


In [ ]:

parent_rows=[]; summary_rows=[]
for k in K_VALUES:
    if len(eligible_keys)<k: continue
    combos,total,mode=sample_combinations(eligible_keys,k,MAX_COMBINATIONS_PER_K)
    summary_rows.append({'k':k,'n_assets_available':len(eligible_keys),'total_possible':total,'generated':len(combos),'generation_mode':mode})
    for combo in combos:
        rec={'parent_instance_id':parent_instance_id(combo,k),'dataset_version':VERSION,'k':k,'asset_keys':'|'.join(combo)}
        for i,a in enumerate(combo): rec[f'asset_key_{i}']=a
        parent_rows.append(rec)
parents_df=pd.DataFrame(parent_rows); generation_summary=pd.DataFrame(summary_rows)
display(generation_summary); print('Parents:',parents_df.shape); display(parents_df.head(20))
if parents_df['parent_instance_id'].duplicated().any(): raise RuntimeError('Duplicação de parent_instance_id')
PARENTS_FILE=DIRS['stage']/f'hamiltonian_parents__{STAGE_NAME}__{VERSION}.parquet'; parents_df.to_parquet(PARENTS_FILE,index=False)
generation_summary.to_csv(DIRS['stage']/f'generation_summary__{STAGE_NAME}__{VERSION}.csv',index=False)


## 8. Split determinístico sem vazamento por parent

In [ ]:

def hash_u01(text): return int(hashlib.sha256(text.encode()).hexdigest()[:16],16)/float(16**16-1)
def split_from_id(pid):
    u=hash_u01(pid)
    return 'train' if u<TRAIN_FRAC else ('calibration' if u<TRAIN_FRAC+CAL_FRAC else 'test')
parents_df['split']=parents_df['parent_instance_id'].map(split_from_id)
display(parents_df.groupby(['k','split']).size().reset_index(name='count'))
n_dupe_across=parents_df.groupby('parent_instance_id')['split'].nunique().gt(1).sum(); print('Leakage de parent:',n_dupe_across); assert n_dupe_across==0
for split in ['train','calibration','test']:
    out=DIRS['stage']/f'hamiltonian_parents__{split}__{STAGE_NAME}__{VERSION}.parquet'; parents_df.loc[parents_df['split']==split].to_parquet(out,index=False); print(split,'->',out.resolve())


In [ ]:

scenario_df=parents_df[['parent_instance_id','dataset_version','k','asset_keys','split']].copy(); scenario_df['scenario_family']='nominal'; scenario_df['scenario_uid']=scenario_df['parent_instance_id'].map(lambda pid: stable_hash(f'{pid}|scenario=nominal',20))
scenario_df=scenario_df[['scenario_uid','parent_instance_id','dataset_version','k','asset_keys','scenario_family','split']]
SCENARIO_FILE=DIRS['stage']/f'nominal_scenarios__{STAGE_NAME}__{VERSION}.parquet'; scenario_df.to_parquet(SCENARIO_FILE,index=False)
print('Scenarios:',scenario_df.shape,'->',SCENARIO_FILE.resolve()); display(scenario_df.head(20))


## 9. Auditoria de sobreposição de ativos entre splits

In [ ]:

assets_by_split={}
for split in ['train','calibration','test']:
    aset=set()
    for s in parents_df.loc[parents_df['split']==split,'asset_keys']: aset.update(str(s).split('|'))
    assets_by_split[split]=aset
rows=[]
for a,b in [('train','calibration'),('train','test'),('calibration','test')]: rows.append({'split_a':a,'split_b':b,'n_assets_a':len(assets_by_split[a]),'n_assets_b':len(assets_by_split[b]),'n_overlap':len(assets_by_split[a]&assets_by_split[b]),'overlap_assets':'|'.join(sorted(assets_by_split[a]&assets_by_split[b]))})
asset_overlap_df=pd.DataFrame(rows); display(asset_overlap_df); asset_overlap_df.to_csv(DIRS['stage']/f'asset_overlap_audit__{STAGE_NAME}__{VERSION}.csv',index=False)


## 10. Auditoria final e contrato permanente de caminhos

In [ ]:

checks=[]
def add_check(name,passed,detail): checks.append({'check':name,'passed':bool(passed),'detail':str(detail)})
for name in ['registry','raw','processed','splits','instances','manifests']: add_check(f'{name}_dir_exists',DIRS[name].exists(),DIRS[name])
add_check('registry_rows_positive',len(registry_df)>0,len(registry_df)); add_check('raw_files_positive',len(raw_files)>0,len(raw_files)); add_check('parent_ids_unique',not parents_df['parent_instance_id'].duplicated().any(),len(parents_df)); add_check('scenario_ids_unique',not scenario_df['scenario_uid'].duplicated().any(),len(scenario_df)); add_check('no_parent_split_leakage',n_dupe_across==0,n_dupe_across); add_check('schema_manifest_written',SCHEMA_MANIFEST_FILE.exists(),SCHEMA_MANIFEST_FILE); add_check('path_manifest_written',PATH_MANIFEST_FILE.exists(),PATH_MANIFEST_FILE); add_check('canonical_table_written',CANONICAL_FILE.exists(),CANONICAL_FILE); add_check('parents_written',PARENTS_FILE.exists(),PARENTS_FILE); add_check('scenarios_written',SCENARIO_FILE.exists(),SCENARIO_FILE)
check_df=pd.DataFrame(checks); display(check_df)
FINAL_AUDIT_FILE=DIRS['stage']/f'final_audit__{STAGE_NAME}__{VERSION}.csv'; check_df.to_csv(FINAL_AUDIT_FILE,index=False)
if not check_df['passed'].all(): raise RuntimeError('Falhas:
'+check_df.loc[~check_df['passed']].to_string(index=False))
print('TODOS OS CHECKS OBRIGATÓRIOS PASSARAM.')


In [ ]:

contract={'dataset_root':str(ROOT.resolve()),'registry_dir':str(DIRS['registry'].resolve()),'raw_dir':str(DIRS['raw'].resolve()),'processed_dir':str(DIRS['processed'].resolve()),'splits_dir':str(DIRS['splits'].resolve()),'instances_dir':str(DIRS['instances'].resolve()),'manifests_dir':str(DIRS['manifests'].resolve()),'stage_20_21B_dir':str(DIRS['stage'].resolve()),'canonical_asset_table':str(CANONICAL_FILE.resolve()),'schema_manifest':str(SCHEMA_MANIFEST_FILE.resolve()),'path_manifest':str(PATH_MANIFEST_FILE.resolve()),'hamiltonian_parents':str(PARENTS_FILE.resolve()),'nominal_scenarios':str(SCENARIO_FILE.resolve())}
CONTRACT_FILE=DIRS['stage']/f'path_contract__{STAGE_NAME}__{VERSION}.json'
with open(CONTRACT_FILE,'w',encoding='utf-8') as f: json.dump(contract,f,indent=2,ensure_ascii=False)
print(json.dumps(contract,indent=2,ensure_ascii=False)); print('Contrato salvo:',CONTRACT_FILE.resolve())


## Critério de encerramento

A 20.21B só deve ser considerada concluída quando a última célula confirmar todos os checks. Os notebooks seguintes devem consumir o `path_contract`, `path_manifest` e `schema_manifest`, evitando procurar caminhos ou adivinhar colunas novamente.